# XGBoost Evaluation on LIAR

This notebook evaluates the saved text-only XGBoost model on the imported LIAR dataset. The new `content` column is processed with the same preprocessing functions used in the main cleaning pipeline. No split is performed here.

In [1]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import joblib
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix

from src.preprocessing import normalize_text, process_and_tokenize, reset_vocab_tracking
from src.xgboost_features import transform_text_in_chunks

tfidf = joblib.load('../models/tfidf_vectorizer1500ot.joblib')

xgb_model = xgb.XGBClassifier()
xgb_model.load_model('../models/xgboost_model1500ot.json')

In [2]:
liar_columns = [
    'id', 'label', 'statement', 'subject', 'speaker', 'job_title',
    'state', 'party', 'barely_true_counts', 'false_counts',
    'half_true_counts', 'mostly_true_counts', 'pants_fire_counts', 'context'
]

df_train = pd.read_csv('../data/liar-data/liar_dataset/train.tsv', sep='\t', header=None, names=liar_columns)
df_val = pd.read_csv('../data/liar-data/liar_dataset/valid.tsv', sep='\t', header=None, names=liar_columns)
df_test = pd.read_csv('../data/liar-data/liar_dataset/test.tsv', sep='\t', header=None, names=liar_columns)

df_liar_full = pd.concat([df_train, df_val, df_test], ignore_index=True)

label_mapping = {
    'pants-fire': 1,
    'false': 1,
    'barely-true': 1,
    'half-true': 0,
    'mostly-true': 0,
    'true': 0,
}

df_liar_full['target'] = df_liar_full['label'].map(label_mapping)
df_liar_full = df_liar_full.dropna(subset=['target']).copy()
df_liar_full['target'] = df_liar_full['target'].astype(int)

df_liar_full['content'] = df_liar_full['statement'].fillna('')

print(f'Total LIAR rows after label mapping: {len(df_liar_full)}')
df_liar_full[['label', 'target', 'content']].head()

Total LIAR rows after label mapping: 12791


,label,target,content
0,false,1,Says the Annies List political group supports ...
1,half-true,0,When did the decline of coal start? It started...
2,mostly-true,0,"Hillary Clinton agrees with John McCain ""by vo..."
3,false,1,Health care reform legislation is likely to ma...
4,half-true,0,The economic turnaround started at the end of ...


In [3]:
reset_vocab_tracking()

df_liar_full['content_normalized'] = df_liar_full['content'].apply(normalize_text)
df_liar_full['content_processed'] = df_liar_full['content_normalized'].apply(process_and_tokenize)

df_liar_full[['content', 'content_normalized', 'content_processed']].head()

,content,content_normalized,content_processed
0,Says the Annies List political group supports ...,says the annies list political group supports ...,say anni list polit group support third trimes...
1,When did the decline of coal start? It started...,when did the decline of coal start it started ...,declin coal start start natur gas took start b...
2,"Hillary Clinton agrees with John McCain ""by vo...",hillary clinton agrees with john mccain by vot...,hillari clinton agre john mccain vote give geo...
3,Health care reform legislation is likely to ma...,health care reform legislation is likely to ma...,health care reform legisl like mandat free sex...
4,The economic turnaround started at the end of ...,the economic turnaround started at the end of ...,econom turnaround start end term


In [4]:
X_liar_final = transform_text_in_chunks(
    df_liar_full['content_processed'],
    tfidf,
    chunk_size=50000,
    label='liar',
)
y_liar = df_liar_full['target'].values

print(f'Final LIAR matrix shape: {X_liar_final.shape}')

TF-IDF transform on liar rows 0 to 12,791...
Final LIAR matrix shape: (12791, 1500)


In [5]:
y_liar_pred = xgb_model.predict(X_liar_final)

print('Classification Report:')
print(classification_report(y_liar, y_liar_pred, target_names=['Reliable/True-ish (0)', 'Fake-ish (1)']))
print('Confusion Matrix:')
print(confusion_matrix(y_liar, y_liar_pred))

Classification Report:
                       precision    recall  f1-score   support

Reliable/True-ish (0)       0.62      0.35      0.44      7134
         Fake-ish (1)       0.47      0.73      0.57      5657

             accuracy                           0.52     12791
            macro avg       0.54      0.54      0.51     12791
         weighted avg       0.55      0.52      0.50     12791

Confusion Matrix:
[[2474 4660]
 [1520 4137]]
